# NHANES DPQ(PHQ-9) Quickstart — Recreated

연구 디렉터리 한정. 추가 병합 없이 `DPQ_L.xpt`만 사용합니다.


In [42]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
DPQ_FILE = DATA_DIR / 'DPQ_L.xpt'
assert DPQ_FILE.exists(), f"DPQ file not found: {DPQ_FILE}"

dpq = pd.read_sas(DPQ_FILE, format='xport')
print('loaded shape:', dpq.shape)
dpq.head(3)


loaded shape: (6337, 11)


,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090,DPQ100
0,130378.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,130379.0,5.397605e-79,5.397605e-79,1.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
2,130380.0,5.397605e-79,5.397605e-79,1.0,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79


In [43]:
# 점수·라벨 생성
VALID = {0,1,2,3}
items = [f'DPQ0{n:02d}' for n in range(10, 100, 10)]
cols = {c.decode() if isinstance(c, bytes) else c: c for c in dpq.columns}
dpq.rename(columns=cols, inplace=True)

def clean_item(s):
    s = pd.to_numeric(s, errors='coerce')
    return s.where(s.isin(VALID), np.nan)

for it in items:
    if it in dpq.columns:
        dpq[it] = clean_item(dpq[it])
    else:
        dpq[it] = np.nan

dpq['phq9_sum'] = dpq[items].sum(axis=1, min_count=1)
dpq['phq9_missing'] = dpq[items].isna().sum(axis=1)

def severity(score):
    if pd.isna(score):
        return np.nan
    s = int(score)
    if s <= 4: return 'none'
    if s <= 9: return 'mild'
    if s <= 14: return 'moderate'
    if s <= 19: return 'moderately_severe'
    return 'severe'

dpq['phq9_severity'] = dpq['phq9_sum'].apply(severity)
dpq['needsAttention'] = (dpq['phq9_sum'] >= 10).astype('boolean')

dpq[['SEQN','phq9_sum','phq9_severity','needsAttention','phq9_missing']].head(10)


,SEQN,phq9_sum,phq9_severity,needsAttention,phq9_missing
0,130378.0,NaN,NaN,False,9
1,130379.0,1.0,none,False,8
2,130380.0,2.0,none,False,7
3,130386.0,1.0,none,False,8
4,130387.0,NaN,NaN,False,9
5,130388.0,NaN,NaN,False,9
6,130389.0,NaN,NaN,False,9
7,130390.0,NaN,NaN,False,9
8,130391.0,24.0,severe,True,0
9,130392.0,1.0,none,False,8


In [44]:
# 재현성: 파일 해시
import hashlib

def sha256sum(path):
    with open(path, 'rb') as f:
        h = hashlib.sha256()
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
        return h.hexdigest()

print('DPQ_L.xpt SHA-256 =', sha256sum(DPQ_FILE))


DPQ_L.xpt SHA-256 = 25605a02685035fe997477b31a0991cca2776b0f72f24a8e61ac5b018456304b


In [45]:
# 견고한 분할(계층화 보강 포함)
from collections import Counter
from sklearn.model_selection import train_test_split

valid_mask = dpq['phq9_missing'].fillna(9).astype(int).eq(0) & dpq['phq9_sum'].notna()
subset = dpq.loc[valid_mask, ['SEQN','phq9_sum','phq9_severity','needsAttention']].copy()

def try_stratified_split(df, labels, test_size=0.2, seed=20250927):
    counts = Counter(labels)
    if min(counts.values()) < 2:
        raise ValueError('min class count < 2')
    idx_train, idx_test = train_test_split(
        df.index, test_size=test_size, random_state=seed, stratify=labels
    )
    return df.loc[idx_train].reset_index(drop=True), df.loc[idx_test].reset_index(drop=True)

labels = subset['phq9_severity']
try:
    train_df, test_df = try_stratified_split(subset, labels)
except Exception:
    map2 = {'severe':'moderately_severe'}
    labels2 = labels.replace(map2)
    try:
        train_df, test_df = try_stratified_split(subset, labels2)
    except Exception:
        map3 = {'severe':'moderately_severe','none':'mild'}
        labels3 = labels.replace(map3)
        try:
            train_df, test_df = try_stratified_split(subset, labels3)
        except Exception:
            idx_train, idx_test = train_test_split(subset.index, test_size=0.2, random_state=20250927)
            train_df = subset.loc[idx_train].reset_index(drop=True)
            test_df = subset.loc[idx_test].reset_index(drop=True)

len(train_df), len(test_df), train_df['phq9_severity'].value_counts(), test_df['phq9_severity'].value_counts()


(93,
 24,
 phq9_severity
 severe               36
 moderately_severe    34
 moderate             22
 mild                  1
 Name: count, dtype: int64,
 phq9_severity
 moderately_severe    10
 severe               10
 moderate              4
 Name: count, dtype: int64)

In [46]:
# 분할 결과 저장
from pathlib import Path
out_dir = Path('../outputs/splits')
out_dir.mkdir(parents=True, exist_ok=True)
train_df.to_csv(out_dir / 'train.csv', index=False)
test_df.to_csv(out_dir / 'test.csv', index=False)
print('saved splits to', out_dir)


saved splits to ..\outputs\splits


## 예측 스키마 및 베이스라인


- 예측 CSV: `../outputs/predictions/test_predictions.csv`
- 컬럼: `SEQN`, `pred_riskLevel{none|mild|moderate|moderately_severe|severe}`, `pred_needsAttention{true|false}`
- 샘플: `../outputs/predictions/test_predictions.sample.csv`


In [47]:
# 베이스라인 예측 생성(라벨 복사)
from pathlib import Path
pred_dir = Path('../outputs/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

baseline = test_df[['SEQN','phq9_severity','needsAttention']].copy()
baseline.rename(columns={'phq9_severity':'pred_riskLevel','needsAttention':'pred_needsAttention'}, inplace=True)

out_path = pred_dir / 'test_predictions.csv'
baseline.to_csv(out_path, index=False)
print('saved baseline predictions:', out_path)


saved baseline predictions: ..\outputs\predictions\test_predictions.csv


## 평가


In [48]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

pred_path = Path('../outputs/predictions/test_predictions.csv')
if pred_path.exists():
    preds = pd.read_csv(pred_path)
    merged = test_df.merge(preds, on='SEQN', how='inner')
    merged['pred_needsAttention'] = merged['pred_needsAttention'].astype(str).str.lower().map({'true': True, 'false': False}).fillna(merged['pred_needsAttention'])

    # riskLevel
    y_true_rl = merged['phq9_severity']
    y_pred_rl = merged['pred_riskLevel']
    rl_acc = accuracy_score(y_true_rl, y_pred_rl)
    rl_f1 = f1_score(y_true_rl, y_pred_rl, average='macro')
    rl_cm = confusion_matrix(y_true_rl, y_pred_rl, labels=['none','mild','moderate','moderately_severe','severe'])
    print('[riskLevel] accuracy=', rl_acc, ' macro-F1=', rl_f1)
    print('CM:\n', rl_cm)
    print(classification_report(y_true_rl, y_pred_rl, digits=3))

    # needsAttention
    y_true_na = merged['needsAttention'].astype(bool)
    y_pred_na = merged['pred_needsAttention'].astype(bool)
    na_acc = accuracy_score(y_true_na, y_pred_na)
    na_f1 = f1_score(y_true_na, y_pred_na)
    print('[needsAttention] accuracy=', na_acc, ' F1=', na_f1)
else:
    print('예측 파일이 없습니다:', pred_path)


[riskLevel] accuracy= 1.0  macro-F1= 1.0
CM:
 [[ 0  0  0  0  0]
 [ 0  0  0  0  0]
 [ 0  0  4  0  0]
 [ 0  0  0 10  0]
 [ 0  0  0  0 10]]
                   precision    recall  f1-score   support

         moderate      1.000     1.000     1.000         4
moderately_severe      1.000     1.000     1.000        10
           severe      1.000     1.000     1.000        10

         accuracy                          1.000        24
        macro avg      1.000     1.000     1.000        24
     weighted avg      1.000     1.000     1.000        24

[needsAttention] accuracy= 1.0  F1= 1.0


## OpenAI 배치 추론(프로젝트 설정과 동일)
- 키: 환경변수 `OPENAI_API_KEY` 사용
- 엔드포인트: `https://api.openai.com/v1/chat/completions` (또는 `OPENAI_API_URL`로 재정의)
- 모델/파라미터: model=gpt-5-mini, temperature=0.3, response_format=json_object, thinking=true, reasoning_effort=medium, verbosity=medium
- 출력: `../outputs/predictions/test_predictions.csv`


In [49]:
def call_openai(prompt):
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {API_KEY}'
    }
    body = {
        'model': 'gpt-5-mini',
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': 1800,
        'temperature': 0.3,
        'response_format': {'type': 'json_object'},
        'thinking': True,
        'reasoning_effort': 'medium',
        'verbosity': 'medium'
    }
    
    try:
        r = requests.post(API_URL, headers=headers, json=body, timeout=120)
        r.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"Error response: {e.response.text}")  # 상세 에러 메시지 출력
        raise
    
    # 나머지 처리...

In [50]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
DPQ_FILE = DATA_DIR / 'DPQ_L.xpt'

assert DPQ_FILE.exists(), f"DPQ file not found: {DPQ_FILE}"

dpq = pd.read_sas(DPQ_FILE, format='xport')
print(dpq.shape)
dpq.head(3)


(6337, 11)


,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090,DPQ100
0,130378.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,130379.0,5.397605e-79,5.397605e-79,1.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
2,130380.0,5.397605e-79,5.397605e-79,1.0,1.000000e+00,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79


In [51]:
# 유효값(0,1,2,3)만 사용하여 9문항 점수 합산
VALID = {0,1,2,3}
items = [f'DPQ0{n:02d}' for n in range(10, 100, 10)]  # DPQ010..DPQ090

# 일부 환경에서 컬럼이 bytes로 로드될 수 있어 문자열 정규화
cols = {c.decode() if isinstance(c, bytes) else c: c for c in dpq.columns}
dpq.rename(columns=cols, inplace=True)

def clean_item(s):
    s = pd.to_numeric(s, errors='coerce')
    return s.where(s.isin(VALID), np.nan)

for it in items:
    if it in dpq.columns:
        dpq[it] = clean_item(dpq[it])
    else:
        dpq[it] = np.nan

# 총점과 결측 개수
dpq['phq9_sum'] = dpq[items].sum(axis=1, min_count=1)
dpq['phq9_missing'] = dpq[items].isna().sum(axis=1)

def severity(score):
    if pd.isna(score):
        return np.nan
    s = int(score)
    if s <= 4: return 'none'
    if s <= 9: return 'mild'
    if s <= 14: return 'moderate'
    if s <= 19: return 'moderately_severe'
    return 'severe'

dpq['phq9_severity'] = dpq['phq9_sum'].apply(severity)

# needsAttention 규칙(예: 총점≥10)
dpq['needsAttention'] = (dpq['phq9_sum'] >= 10).astype('boolean')

dpq[['SEQN','phq9_sum','phq9_severity','needsAttention','phq9_missing']].head(10)


,SEQN,phq9_sum,phq9_severity,needsAttention,phq9_missing
0,130378.0,NaN,NaN,False,9
1,130379.0,1.0,none,False,8
2,130380.0,2.0,none,False,7
3,130386.0,1.0,none,False,8
4,130387.0,NaN,NaN,False,9
5,130388.0,NaN,NaN,False,9
6,130389.0,NaN,NaN,False,9
7,130390.0,NaN,NaN,False,9
8,130391.0,24.0,severe,True,0
9,130392.0,1.0,none,False,8


In [56]:
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {API_KEY}'
}
print("Headers:", {k: v[:20] + "..." if k == 'Authorization' else v for k, v in headers.items()})

Headers: {'Content-Type': 'application/json', 'Authorization': 'Bearer sk-proj-63i4k...'}


In [ ]:
# 하드코딩 Authorization 버전으로 call_openai 재정의(테스트용)
import json, requests

def call_openai(prompt: str):
    headers = {
        'Content-Type': 'application/json',
        'Authorization': 'Bearer OPENAI_API_KEY'
    }
    body = {
        'model': 'gpt-5-mini',
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt}
        ],
        'response_format': {'type': 'json_object'}
    }
    r = requests.post(API_URL, headers=headers, json=body, timeout=120)
    r.raise_for_status()
    data = r.json()
    content = data['choices'][0]['message']['content']
    try:
        return json.loads(content)
    except Exception:
        if '```json' in content:
            start = content.find('```json') + 7
            end = content.rfind('```')
            return json.loads(content[start:end].strip())
        raise


In [58]:
# 간단 요약 통계
summary = dpq[['phq9_sum','phq9_severity','needsAttention']].copy()
print('N =', len(summary))
print('유효 점수 N =', summary['phq9_sum'].notna().sum())
print(summary['phq9_severity'].value_counts(dropna=False))
print(summary['needsAttention'].value_counts(dropna=False))


N = 6337
유효 점수 N = 4178
phq9_severity
none                 2334
NaN                  2159
mild                 1111
moderate              462
moderately_severe     191
severe                 80
Name: count, dtype: int64
needsAttention
False    5604
True      733
Name: count, dtype: Int64


## 다음 단계(선택)
- `DPQ100`(기능장애) 활용한 보조 규칙: `needsAttention |= DPQ100 >= 2`
- 프로젝트 스키마 매핑: `riskLevel ← phq9_severity`, `needsAttention` 유지
- 모델 평가 연계: OpenAI 응답의 `riskLevel/needsAttention`와 비교 메트릭 산출


In [59]:
# 재현성: 입력 파일 해시 기록
import hashlib

def sha256sum(path):
    with open(path, 'rb') as f:
        h = hashlib.sha256()
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
        return h.hexdigest()

file_hash = sha256sum(DPQ_FILE)
print('DPQ_L.xpt SHA-256 =', file_hash)


DPQ_L.xpt SHA-256 = 25605a02685035fe997477b31a0991cca2776b0f72f24a8e61ac5b018456304b


In [60]:
# 내보내기
from pathlib import Path
out_dir = Path('../outputs/splits')
out_dir.mkdir(parents=True, exist_ok=True)
train_df.to_csv(out_dir / 'train.csv', index=False)
test_df.to_csv(out_dir / 'test.csv', index=False)
print('saved to', out_dir)


saved to ..\outputs\splits


## 평가 하네스(플레이스홀더)
- 이 노트북에서는 현재 라벨만 고정합니다.
- 추후 모델 예측 CSV를 `../outputs/predictions/test_predictions.csv`로 저장하면,
  이를 불러와 라벨(`phq9_severity`, `needsAttention`)과 비교하는 셀을 추가합니다.


## 예측 CSV 스키마
- 경로: `../outputs/predictions/test_predictions.csv`
- 열:
  - `SEQN` (int): 테스트셋 키(조인용)
  - `pred_riskLevel` (str): one of {`none`,`mild`,`moderate`,`moderately_severe`,`severe`}
  - `pred_needsAttention` (bool or {`true`,`false`}): 이진 예측
- 주의: 테스트셋(`../outputs/splits/test.csv`)에 존재하는 `SEQN`만 포함하십시오.


In [61]:
# 평가: 예측 불러오기 및 지표 산출
from pathlib import Path
pred_path = Path('../outputs/predictions/test_predictions.csv')

if pred_path.exists():
    preds = pd.read_csv(pred_path)
    merged = test_df.merge(preds, on='SEQN', how='inner')

    # 형식 정리
    merged['pred_needsAttention'] = merged['pred_needsAttention'].astype(str).str.lower().map({'true': True, 'false': False}).fillna(merged['pred_needsAttention'])

    # riskLevel 지표
    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
    y_true_rl = merged['phq9_severity']
    y_pred_rl = merged['pred_riskLevel']
    rl_acc = accuracy_score(y_true_rl, y_pred_rl)
    rl_f1 = f1_score(y_true_rl, y_pred_rl, average='macro')
    rl_cm = confusion_matrix(y_true_rl, y_pred_rl, labels=['none','mild','moderate','moderately_severe','severe'])

    print('[riskLevel] accuracy=', rl_acc, ' macro-F1=', rl_f1)
    print('confusion matrix (rows=true, cols=pred):\n', rl_cm)
    print(classification_report(y_true_rl, y_pred_rl, digits=3))

    # needsAttention 지표
    y_true_na = merged['needsAttention'].astype(bool)
    y_pred_na = merged['pred_needsAttention'].astype(bool)
    na_acc = accuracy_score(y_true_na, y_pred_na)
    na_f1 = f1_score(y_true_na, y_pred_na)

    print('[needsAttention] accuracy=', na_acc, ' F1=', na_f1)
else:
    print('예측 파일이 없습니다:', pred_path)


[riskLevel] accuracy= 1.0  macro-F1= 1.0
confusion matrix (rows=true, cols=pred):
 [[ 0  0  0  0  0]
 [ 0  0  0  0  0]
 [ 0  0  4  0  0]
 [ 0  0  0 10  0]
 [ 0  0  0  0 10]]
                   precision    recall  f1-score   support

         moderate      1.000     1.000     1.000         4
moderately_severe      1.000     1.000     1.000        10
           severe      1.000     1.000     1.000        10

         accuracy                          1.000        24
        macro avg      1.000     1.000     1.000        24
     weighted avg      1.000     1.000     1.000        24

[needsAttention] accuracy= 1.0  F1= 1.0


In [62]:
from collections import Counter

def try_stratified_split(df, labels, test_size=0.2, seed=20250927):
    from sklearn.model_selection import train_test_split
    counts = Counter(labels)
    if min(counts.values()) < 2:
        raise ValueError('min class count < 2')
    idx_train, idx_test = train_test_split(
        df.index, test_size=test_size, random_state=seed, stratify=labels
    )
    return df.loc[idx_train].reset_index(drop=True), df.loc[idx_test].reset_index(drop=True)

# 1) 원라벨 시도
labels = subset['phq9_severity']
try:
    train_df, test_df = try_stratified_split(subset, labels)
except Exception as e1:
    print('원라벨 계층화 실패:', e1)
    # 2) severe -> moderately_severe 합침
    map2 = {'severe':'moderately_severe'}
    labels2 = labels.replace(map2)
    try:
        train_df, test_df = try_stratified_split(subset, labels2)
    except Exception as e2:
        print('합침1 실패:', e2)
        # 3) none -> mild 합침
        map3 = {'severe':'moderately_severe','none':'mild'}
        labels3 = labels.replace(map3)
        try:
            train_df, test_df = try_stratified_split(subset, labels3)
        except Exception as e3:
            print('합침2 실패:', e3)
            # 4) 최종 무작위 분할
            from sklearn.model_selection import train_test_split
            idx_train, idx_test = train_test_split(
                subset.index, test_size=0.2, random_state=20250927
            )
            train_df = subset.loc[idx_train].reset_index(drop=True)
            test_df = subset.loc[idx_test].reset_index(drop=True)

len(train_df), len(test_df), train_df['phq9_severity'].value_counts(), test_df['phq9_severity'].value_counts()

원라벨 계층화 실패: min class count < 2
합침1 실패: min class count < 2
합침2 실패: min class count < 2


(93,
 24,
 phq9_severity
 severe               36
 moderately_severe    34
 moderate             22
 mild                  1
 Name: count, dtype: int64,
 phq9_severity
 moderately_severe    10
 severe               10
 moderate              4
 Name: count, dtype: int64)

In [63]:
# 베이스라인 예측 생성(규칙 기반) → 평가 파일 저장
from pathlib import Path
pred_dir = Path('../outputs/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

baseline = test_df[['SEQN','phq9_severity','needsAttention']].copy()
baseline.rename(columns={
    'phq9_severity':'pred_riskLevel',
    'needsAttention':'pred_needsAttention'
}, inplace=True)

out_path = pred_dir / 'test_predictions.csv'
baseline.to_csv(out_path, index=False)
print('saved predictions to', out_path, 'rows=', len(baseline))

saved predictions to ..\outputs\predictions\test_predictions.csv rows= 24


## 캐시 무시: 최소 필드 OpenAI 호출(gpt-5-mini)
- 백엔드 캐시와 무관하게 직접 REST 호출
- 전송 필드 최소화: `model`, `messages`, `response_format` (옵션으로 `max_completion_tokens` 한 줄)
- 결과 저장: `../outputs/predictions/test_predictions.csv`


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your api key"
os.environ["OPENAI_API_URL"] = "https://api.openai.com/v1/chat/completions"

In [70]:
import json, time, requests
from pathlib import Path

API_URL = os.getenv('OPENAI_API_URL', 'https://api.openai.com/v1/chat/completions')
API_KEY = os.getenv('OPENAI_API_KEY')

SYSTEM_PROMPT = '당신은 노인 요양원의 의료진을 도와주는 AI 의료 분석 전문가입니다.'

headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {API_KEY}',
}

pred_rows = []
for _, row in test_df.iterrows():
    prompt = (
        '지난 2주의 우울 증상 설문(PHQ-9) 응답을 기반으로, 다음 두 값을 JSON으로만 출력하세요.\n'
        '필드: riskLevel{none|mild|moderate|moderately_severe|severe}, needsAttention{true|false}.\n'
        f"총점(0-27): {int(row['phq9_sum']) if not pd.isna(row['phq9_sum']) else 'NA'}"
    )
    body = {
        'model': 'gpt-5-mini',
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt}
        ],
        'response_format': {'type': 'json_object'},
         'max_completion_tokens': 1800,  # 필요 시 활성화
    }
    try:
        r = requests.post(API_URL, headers=headers, json=body, timeout=120)
        r.raise_for_status()
        content = r.json()['choices'][0]['message']['content']
        try:
            resp = json.loads(content)
        except Exception:
            if '```json' in content:
                start = content.find('```json') + 7
                end = content.rfind('```')
                resp = json.loads(content[start:end].strip())
            else:
                raise
        risk = str(resp.get('riskLevel', '')).strip().lower()
        attn = resp.get('needsAttention', None)
        pred_rows.append({'SEQN': row['SEQN'], 'pred_riskLevel': risk, 'pred_needsAttention': bool(attn) if isinstance(attn, bool) else str(attn).lower()=='true'})
    except Exception as e:
        pred_rows.append({'SEQN': row['SEQN'], 'pred_riskLevel': '', 'pred_needsAttention': ''})
        print('error on SEQN', row['SEQN'], e)
    time.sleep(0.5)

pred_df = pd.DataFrame(pred_rows)
pred_path = Path('../outputs/predictions/test_predictions.csv')
pred_df.to_csv(pred_path, index=False)
print('saved OpenAI predictions(minimal body):', pred_path, 'rows=', len(pred_df))


saved OpenAI predictions(minimal body): ..\outputs\predictions\test_predictions.csv rows= 24


In [71]:
# 최종 평가: test_predictions.csv 기준 지표 산출
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

pred_path = Path('../outputs/predictions/test_predictions.csv')
assert pred_path.exists(), f'예측 파일이 없습니다: {pred_path}'

preds = pd.read_csv(pred_path)
merged = test_df.merge(preds, on='SEQN', how='inner')

# 형식 정리 및 유효성 필터
merged['pred_riskLevel'] = merged['pred_riskLevel'].astype(str).str.strip().str.lower()
merged['pred_needsAttention'] = merged['pred_needsAttention'].astype(str).str.lower().map({'true': True, 'false': False})

# riskLevel 지표
labels = ['none','mild','moderate','moderately_severe','severe']
y_true_rl = merged['phq9_severity']
y_pred_rl = merged['pred_riskLevel']
rl_acc = accuracy_score(y_true_rl, y_pred_rl)
rl_f1 = f1_score(y_true_rl, y_pred_rl, average='macro')
rl_cm = confusion_matrix(y_true_rl, y_pred_rl, labels=labels)
print('[riskLevel] accuracy=', rl_acc, ' macro-F1=', rl_f1)
print('confusion matrix (rows=true, cols=pred):\n', rl_cm)
print(classification_report(y_true_rl, y_pred_rl, labels=labels, digits=3, zero_division=0))

# needsAttention 지표
y_true_na = merged['needsAttention'].astype(bool)
y_pred_na = merged['pred_needsAttention'].astype(bool)
na_acc = accuracy_score(y_true_na, y_pred_na)
na_f1 = f1_score(y_true_na, y_pred_na)
print('[needsAttention] accuracy=', na_acc, ' F1=', na_f1)


[riskLevel] accuracy= 1.0  macro-F1= 1.0
confusion matrix (rows=true, cols=pred):
 [[ 0  0  0  0  0]
 [ 0  0  0  0  0]
 [ 0  0  4  0  0]
 [ 0  0  0 10  0]
 [ 0  0  0  0 10]]
                   precision    recall  f1-score   support

             none      0.000     0.000     0.000         0
             mild      0.000     0.000     0.000         0
         moderate      1.000     1.000     1.000         4
moderately_severe      1.000     1.000     1.000        10
           severe      1.000     1.000     1.000        10

         accuracy                          1.000        24
        macro avg      0.600     0.600     0.600        24
     weighted avg      1.000     1.000     1.000        24

[needsAttention] accuracy= 1.0  F1= 1.0
